# Voice-V2 — XTTS v2 fine-tune on Colab

Before running:
1. **Runtime → Change runtime type → T4 GPU** (or A100 if you have Colab Pro).
2. **Upload your dataset** to Drive at `MyDrive/voice-v2/dataset/` so it contains `wavs/` and `metadata.csv` (~21 MB, fits in any free Drive).
3. Run cells top to bottom.

**Storage layout:**
- Dataset stays on Drive (small, persistent).
- All training output (checkpoints ~5 GB) lives on Colab's local disk at `/content/run/` — not Drive. This avoids eating Drive quota.
- The final `best_model.pth` is the only thing we copy to Drive at the end (~2 GB). Or download it directly to your Mac via the file panel.

In [ ]:
# 1. Confirm GPU
!nvidia-smi

In [ ]:
# 2. Mount Google Drive (read-only access to dataset)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Configure paths. Dataset = Drive (persistent). Output = Colab local disk.
DRIVE_BASE = '/content/drive/MyDrive/voice-v2'
DATASET_PATH = f'{DRIVE_BASE}/dataset'
OUTPUT_PATH = '/content/run'  # Colab local disk — ~100 GB free, ephemeral

import os
assert os.path.exists(f'{DATASET_PATH}/metadata.csv'), f'metadata.csv not found in {DATASET_PATH}'
assert os.path.isdir(f'{DATASET_PATH}/wavs'), f'wavs/ folder not found in {DATASET_PATH}'
os.makedirs(OUTPUT_PATH, exist_ok=True)
print('Dataset:', DATASET_PATH)
print('Output (Colab local):', OUTPUT_PATH)
print('Clips:', len(os.listdir(f'{DATASET_PATH}/wavs')))

# Free disk on Colab
!df -h /content | tail -1

In [ ]:
# 4. Clone repo (contains scripts/finetune_xtts.py)
%cd /content
!git clone https://github.com/sforslime/Voice-V2.git || (cd Voice-V2 && git pull)
%cd /content/Voice-V2

In [ ]:
# 5. Install deps (~3 min). Pin transformers<5 (XTTS imports break on 5.x).
!pip install -q 'coqui-tts[codec]' 'transformers<5'

In [ ]:
# 6. Train. Adjust EPOCHS / BATCH_SIZE for your GPU.
#    T4 (16GB): BATCH_SIZE=3 is safe. A100 (40GB): try BATCH_SIZE=6.
#    EPOCHS=20 on 117 clips ≈ 740 steps → ~10–20 min on T4.
import os
os.environ['DATASET_PATH'] = DATASET_PATH
os.environ['OUTPUT_PATH'] = OUTPUT_PATH
os.environ['BATCH_SIZE'] = '3'
os.environ['GRAD_ACCUM'] = '16'
os.environ['EPOCHS'] = '20'
os.environ['SAVE_STEP'] = '500'
os.environ['COQUI_TOS_AGREED'] = '1'

!python scripts/finetune_xtts.py

## Inference from the fine-tuned checkpoint

After training, the checkpoint lives at `/content/run/voice_v2_xtts_ft-*/best_model.pth`. The cell below loads it and synthesizes a sample.

In [ ]:
# 7. Generate sample from fine-tuned model
import glob, os
import torch
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

runs = sorted(glob.glob(f'{OUTPUT_PATH}/voice_v2_xtts_ft-*'))
assert runs, 'No training run found yet.'
run_dir = runs[-1]
ckpt_path = sorted(glob.glob(f'{run_dir}/best_model*.pth'))[-1]
print('Using checkpoint:', ckpt_path)

BASE = f'{OUTPUT_PATH}/xtts_v2_base'
config = XttsConfig()
config.load_json(f'{run_dir}/config.json')
model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_dir=run_dir, vocab_path=f'{BASE}/vocab.json', use_deepspeed=False)
model.cuda()

ref_wav = sorted(glob.glob(f'{DATASET_PATH}/wavs/*.wav'))[0]
out = model.synthesize(
    'Hello, this is my fine-tuned voice. I hope this sounds much more like me now.',
    config, speaker_wav=ref_wav, gpt_cond_len=3, language='en')

import soundfile as sf
OUT_WAV = '/content/finetuned_sample.wav'
sf.write(OUT_WAV, out['wav'], 24000)
from IPython.display import Audio
Audio(OUT_WAV)

## Save the final model

Pick ONE of the two cells below. The fine-tuned checkpoint is ~2 GB.
- **Option A** — copy `best_model.pth` to Drive (needs ~2 GB free in Drive)
- **Option B** — download directly to your Mac (no Drive use, but slow over the browser)

In [ ]:
# 8a. Copy final checkpoint to Drive
import shutil, os
DRIVE_OUT = f'{DRIVE_BASE}/finetuned'
os.makedirs(DRIVE_OUT, exist_ok=True)
shutil.copy(ckpt_path, f'{DRIVE_OUT}/best_model.pth')
shutil.copy(f'{run_dir}/config.json', f'{DRIVE_OUT}/config.json')
shutil.copy(f'{BASE}/vocab.json', f'{DRIVE_OUT}/vocab.json')
print('Saved to', DRIVE_OUT)
!du -sh {DRIVE_OUT}

In [ ]:
# 8b. OR download directly to your Mac (instead of 8a)
from google.colab import files
files.download(ckpt_path)
files.download(f'{run_dir}/config.json')
files.download(f'{BASE}/vocab.json')